# Feature Engineering
Step 1.1: Create Monthly Base Dataset
Objectives

- Aggregate transactions to monthly granularity
- Create complete time index for all branches
- Handle missing periods appropriately

Step 1.2: Weather Feature Engineering
Objectives

- Collect and process weather data
- Calculate HVAC-specific weather features (CDD, HDD)
- Create lagged weather features

Step 1.3: External Data Collection
Objectives

- Add Google Trends data

Step 1.4: Lag & Rolling Features
Objectives

- Create lag features (1, 3, 6, 12 months)
- Create rolling statistics (mean, std, growth rates)
- Create interaction features

In [2]:
import pandas as pd

In [3]:
main_df = pd.read_csv('../data/refactored_df.csv')
weather_df = pd.read_csv('../data/weather_data.csv')
trends_df = pd.read_csv('../data/customer_trends.csv')

In [8]:
# Step 1: Data Preparation and Merging
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Loading datasets...")
print(f"Main dataset shape: {main_df.shape}")
print(f"Weather dataset shape: {weather_df.shape}")
print(f"Trends dataset shape: {trends_df.shape}")

# Convert date columns
main_df['Date'] = pd.to_datetime(main_df['Date'])
main_df = main_df[['Date', 'Branch', 'Segment', 'Rating', 'Tonnage', 'Qty', 'Year', 'Month', 'Week']]
weather_df['Date'] = pd.to_datetime(weather_df['Date'])
trends_df['Month'] = pd.to_datetime(trends_df['Month'])

print("\nDate ranges:")
print(f"Main data: {main_df['Date'].min()} to {main_df['Date'].max()}")
print(f"Weather data: {weather_df['Date'].min()} to {weather_df['Date'].max()}")
print(f"Trends data: {trends_df['Month'].min()} to {trends_df['Month'].max()}")


Loading datasets...
Main dataset shape: (147595, 9)
Weather dataset shape: (456, 11)
Trends dataset shape: (82, 2)

Date ranges:
Main data: 2019-04-03 00:00:00 to 2024-03-30 00:00:00
Weather data: 2019-04-01 00:00:00 to 2025-10-01 00:00:00
Trends data: 2019-01-01 00:00:00 to 2025-10-01 00:00:00


In [37]:
grouped_df = (
    main_df.groupby([main_df['Date'].dt.to_period('M').dt.to_timestamp().rename('MonthStart'), 'Branch'])
    .agg({'Qty': 'sum'})
    .reset_index()
)
grouped_df.columns = ['Date', 'Branch', 'Qty']

In [38]:
grouped_df

,Date,Branch,Qty
0,2019-04-01,BLR,2667.0
1,2019-04-01,COK,1756.0
2,2019-04-01,MAA,13121.5
3,2019-04-01,SBD,3564.5
4,2019-04-01,SBD1,5260.0
...,...,...,...
290,2024-03-01,BLR,7137.0
291,2024-03-01,COK,10894.5
292,2024-03-01,MAA,21649.0
293,2024-03-01,SBD,22408.0


In [48]:
from sklearn.preprocessing import LabelEncoder

In [62]:
monthly_sales = grouped_df.sort_values('Date')
monthly_sales['Month_Date'] = pd.to_datetime(monthly_sales['Date'])
monthly_sales['Year'] = monthly_sales['Month_Date'].dt.year
monthly_sales['Month_Num'] = monthly_sales['Month_Date'].dt.month

le = LabelEncoder()
monthly_sales['Branch'] = le.fit_transform(monthly_sales['Branch'])
print("Encoding Classes:",le.classes_)

print("Monthly sales data prepared:")
print(monthly_sales.head(10))
print(f"\nTotal months in dataset: {len(monthly_sales)}")
print(f"Sales range: {monthly_sales['Qty'].min():.2f} to {monthly_sales['Qty'].max():.2f}")

# Set Month as index for time series
monthly_sales_ts = monthly_sales.set_index('Month_Date')[['Branch', 'Qty']]
print(f"\nTime series prepared with {len(monthly_sales_ts)} monthly observations")

Encoding Classes: ['BLR' 'COK' 'MAA' 'SBD' 'SBD1']
Monthly sales data prepared:
        Date  Branch      Qty Month_Date  Year  Month_Num
0 2019-04-01       0   2667.0 2019-04-01  2019          4
1 2019-04-01       1   1756.0 2019-04-01  2019          4
2 2019-04-01       2  13121.5 2019-04-01  2019          4
3 2019-04-01       3   3564.5 2019-04-01  2019          4
4 2019-04-01       4   5260.0 2019-04-01  2019          4
9 2019-05-01       4  11387.5 2019-05-01  2019          5
7 2019-05-01       2  11896.5 2019-05-01  2019          5
8 2019-05-01       3   5371.0 2019-05-01  2019          5
5 2019-05-01       0   2446.5 2019-05-01  2019          5
6 2019-05-01       1   1349.0 2019-05-01  2019          5

Total months in dataset: 295
Sales range: 22.00 to 23411.00

Time series prepared with 295 monthly observations


In [70]:
# Create time series features for machine learning
def create_time_features(df):
    """Create time-based features for machine learning"""
    features_df = df.copy()
    
    # Time-based features
    features_df['month'] = features_df.index.month
    features_df['quarter'] = features_df.index.quarter
    features_df['year'] = features_df.index.year
    features_df['day_of_year'] = features_df.index.dayofyear
    
    # Cyclical features (sin/cos encoding)
    features_df['month_sin'] = np.sin(2 * np.pi * features_df['month'] / 12)
    features_df['month_cos'] = np.cos(2 * np.pi * features_df['month'] / 12)
    features_df['quarter_sin'] = np.sin(2 * np.pi * features_df['quarter'] / 4)
    features_df['quarter_cos'] = np.cos(2 * np.pi * features_df['quarter'] / 4)
    
    # Lag features (previous months' sales)
    for lag in [1, 2, 3, 6, 12]:
        features_df[f'lag_{lag}'] = features_df['Qty'].shift(lag)
    
    # Rolling statistics (moving averages)
    for window in [3, 6, 12]:
        features_df[f'rolling_mean_{window}'] = features_df['Qty'].shift(1).rolling(window=window).mean()
        features_df[f'rolling_std_{window}'] = features_df['Qty'].shift(1).rolling(window=window).std()
    
    # Trend feature (time since start)
    features_df['time_trend'] = np.arange(len(features_df))
    
    return features_df

# Create features
monthly_features = create_time_features(monthly_sales_ts)
print("Time series features created:")
print(monthly_features.head(10))
print(f"\nFeature columns: {list(monthly_features.columns)}")

Time series features created:
            Branch      Qty  month  quarter  year  day_of_year  month_sin  \
Month_Date                                                                  
2019-04-01       0   2667.0      4        2  2019           91   0.866025   
2019-04-01       1   1756.0      4        2  2019           91   0.866025   
2019-04-01       2  13121.5      4        2  2019           91   0.866025   
2019-04-01       3   3564.5      4        2  2019           91   0.866025   
2019-04-01       4   5260.0      4        2  2019           91   0.866025   
2019-05-01       4  11387.5      5        2  2019          121   0.500000   
2019-05-01       2  11896.5      5        2  2019          121   0.500000   
2019-05-01       3   5371.0      5        2  2019          121   0.500000   
2019-05-01       0   2446.5      5        2  2019          121   0.500000   
2019-05-01       1   1349.0      5        2  2019          121   0.500000   

            month_cos   quarter_sin  quarter_

In [57]:
# Prepare time series data - aggregate monthly sales
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [71]:
monthly_features

,Branch,Qty,month,quarter,year,day_of_year,month_sin,month_cos,quarter_sin,quarter_cos,...,lag_3,lag_6,lag_12,rolling_mean_3,rolling_std_3,rolling_mean_6,rolling_std_6,rolling_mean_12,rolling_std_12,time_trend
Month_Date,,,,,,,,,,,,,,,,,,,,,
2019-04-01,0,2667.0,4,2,2019,91,0.866025,-5.000000e-01,1.224647e-16,-1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2019-04-01,1,1756.0,4,2,2019,91,0.866025,-5.000000e-01,1.224647e-16,-1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2019-04-01,2,13121.5,4,2,2019,91,0.866025,-5.000000e-01,1.224647e-16,-1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2019-04-01,3,3564.5,4,2,2019,91,0.866025,-5.000000e-01,1.224647e-16,-1.000000e+00,...,2667.0,NaN,NaN,5848.166667,6315.339546,NaN,NaN,NaN,NaN,3
2019-04-01,4,5260.0,4,2,2019,91,0.866025,-5.000000e-01,1.224647e-16,-1.000000e+00,...,1756.0,NaN,NaN,6147.333333,6107.120318,NaN,NaN,NaN,NaN,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-03-01,2,21649.0,3,1,2024,61,1.000000,6.123234e-17,1.000000e+00,6.123234e-17,...,6231.0,2871.0,23411.0,5251.000000,1124.041369,7478.750000,4529.984103,8152.708333,5975.911575,290
2024-03-01,3,22408.0,3,1,2024,61,1.000000,6.123234e-17,1.000000e+00,6.123234e-17,...,4024.0,12797.5,6646.0,10390.333333,9778.105662,10608.416667,6684.302869,8005.875000,5575.126697,291
2024-03-01,0,7137.0,3,1,2024,61,1.000000,6.123234e-17,1.000000e+00,6.123234e-17,...,5498.0,13451.0,4325.0,16518.333333,9551.430800,12210.166667,8275.795875,9319.375000,6920.135128,292


In [72]:
# Prepare data for training (remove NaN values from lag/rolling features)
training_data = monthly_features.dropna().copy()
print(f"Training data points after removing NaN: {len(training_data)}")

# Separate features and target
feature_cols = [col for col in training_data.columns if col != 'Qty']
X = training_data[feature_cols]
y = training_data['Qty']

# Split data for time series (use first 80% for training, last 20% for testing)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Training period: {X_train.index[0]} to {X_train.index[-1]}")
print(f"Test period: {X_test.index[0]} to {X_test.index[-1]}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Training data points after removing NaN: 283
Training set: 226 samples
Test set: 57 samples
Training period: 2019-06-01 00:00:00 to 2023-04-01 00:00:00
Test period: 2023-04-01 00:00:00 to 2024-03-01 00:00:00


In [73]:
# Train multiple machine learning models
models = {}
predictions = {}
metrics = {}

# 1. Linear Regression
print("Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

models['Linear Regression'] = lr_model
predictions['Linear Regression'] = lr_pred

# 2. Random Forest
print("Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)  # RF can handle unscaled data
rf_pred = rf_model.predict(X_test)

models['Random Forest'] = rf_model
predictions['Random Forest'] = rf_pred

# 3. Simple trend model (polynomial regression)
print("Training Polynomial Trend...")
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('scaler', StandardScaler()),
    ('linear', LinearRegression())
])

# Use only time trend for polynomial model
X_train_trend = X_train[['time_trend']]
X_test_trend = X_test[['time_trend']]

poly_model.fit(X_train_trend, y_train)
poly_pred = poly_model.predict(X_test_trend)

models['Polynomial Trend'] = poly_model
predictions['Polynomial Trend'] = poly_pred

print("All models trained successfully!")

Training Linear Regression...
Training Random Forest...
Training Polynomial Trend...
All models trained successfully!


In [74]:
# Evaluate model performance
def evaluate_model(y_true, y_pred, model_name):
    """Calculate evaluation metrics for a model"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    # Mean Absolute Percentage Error
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    return {
        'Model': model_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R²': r2,
        'MAPE': mape
    }

# Evaluate all models
results = []
for model_name, pred in predictions.items():
    metrics = evaluate_model(y_test, pred, model_name)
    results.append(metrics)

# Create results DataFrame
results_df = pd.DataFrame(results)
print("Model Performance Comparison:")
print("="*60)
print(results_df.round(2))

# Find best model
best_model_idx = results_df['RMSE'].idxmin()
best_model_name = results_df.loc[best_model_idx, 'Model']
print(f"\nBest performing model: {best_model_name}")
print(f"Best RMSE: {results_df.loc[best_model_idx, 'RMSE']:.2f}")
print(f"Best R²: {results_df.loc[best_model_idx, 'R²']:.3f}")
print(f"Best MAPE: {results_df.loc[best_model_idx, 'MAPE']:.2f}%")

Model Performance Comparison:
               Model      MAE          MSE     RMSE    R²    MAPE
0  Linear Regression  2922.17  22245731.12  4716.54  0.20   48.38
1      Random Forest  2056.68  11570112.35  3401.49  0.59   38.96
2   Polynomial Trend  4083.22  26276900.69  5126.10  0.06  118.38

Best performing model: Random Forest
Best RMSE: 3401.49
Best R²: 0.586
Best MAPE: 38.96%


In [84]:
# Import required libraries for deep learning
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.layers import Input, LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version: {tf.__version__}")
print("Deep learning libraries imported successfully!")

TensorFlow version: 2.20.0
Deep learning libraries imported successfully!


In [86]:
# Prepare data for deep learning models
def create_sequences(data, lookback_window=12, forecast_horizon=1):
    """
    Create sequences for time series prediction
    Args:
        data: Time series data
        lookback_window: Number of previous time steps to use as input
        forecast_horizon: Number of future time steps to predict
    """
    X, y = [], []
    for i in range(lookback_window, len(data) - forecast_horizon + 1):
        X.append(data[i-lookback_window:i])
        y.append(data[i:i+forecast_horizon])
    return np.array(X), np.array(y)

sales_data = monthly_sales_ts

# Scale ONLY the Qty column for deep learning
scaler_dl = MinMaxScaler(feature_range=(0, 1))
qty_scaled = scaler_dl.fit_transform(sales_data[['Qty']])

print(f"Original sales data shape: {sales_data.shape}")
print(f"Qty scaled data shape: {qty_scaled.shape}")
print(f"Scaled Qty range: {qty_scaled.min():.3f} to {qty_scaled.max():.3f}")

# Create sequences for deep learning (12 months lookback, 1 month prediction)
lookback_window = 12
X_sequences, y_sequences = create_sequences(qty_scaled.flatten(), lookback_window, 1)

print(f"Sequences created:")
print(f"X_sequences shape: {X_sequences.shape} (samples, timesteps)")
print(f"y_sequences shape: {y_sequences.shape} (samples, forecast_horizon)")

# Split into train and test for deep learning
# Use same approach as before: 80% train, 20% test
train_size = int(len(X_sequences) * 0.8)
X_train_dl = X_sequences[:train_size]
X_test_dl = X_sequences[train_size:]
y_train_dl = y_sequences[:train_size]
y_test_dl = y_sequences[train_size:]

print(f"\nDeep Learning Data Split:")
print(f"Training samples: {len(X_train_dl)}")
print(f"Test samples: {len(X_test_dl)}")
print(f"Training period: {train_size + lookback_window} months from start")
print(f"Test period: {len(X_test_dl)} months")

Original sales data shape: (295, 2)
Qty scaled data shape: (295, 1)
Scaled Qty range: 0.000 to 1.000
Sequences created:
X_sequences shape: (283, 12) (samples, timesteps)
y_sequences shape: (283, 1) (samples, forecast_horizon)

Deep Learning Data Split:
Training samples: 226
Test samples: 57
Training period: 238 months from start
Test period: 57 months


In [87]:
# Create deep learning models
def create_lstm_model(input_shape):
    """Create LSTM model"""
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(50, return_sequences=False),
        Dropout(0.2),
        Dense(25),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def create_gru_model(input_shape):
    """Create GRU model"""
    model = Sequential([
        GRU(50, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        GRU(50, return_sequences=False),
        Dropout(0.2),
        Dense(25),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def create_cnn_lstm_model(input_shape):
    """Create CNN-LSTM hybrid model"""
    model = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        LSTM(50, return_sequences=False),
        Dropout(0.2),
        Dense(25),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def create_simple_rnn_model(input_shape):
    """Create simple RNN model for comparison"""
    model = Sequential([
        tf.keras.layers.SimpleRNN(50, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        tf.keras.layers.SimpleRNN(50, return_sequences=False),
        Dropout(0.2),
        Dense(25),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Reshape data for deep learning models (add feature dimension)
X_train_dl_reshaped = X_train_dl.reshape((X_train_dl.shape[0], X_train_dl.shape[1], 1))
X_test_dl_reshaped = X_test_dl.reshape((X_test_dl.shape[0], X_test_dl.shape[1], 1))

input_shape = (X_train_dl_reshaped.shape[1], X_train_dl_reshaped.shape[2])
print(f"Input shape for deep learning models: {input_shape}")

# Initialize models
dl_models = {}
dl_models['LSTM'] = create_lstm_model(input_shape)
dl_models['GRU'] = create_gru_model(input_shape)
dl_models['CNN-LSTM'] = create_cnn_lstm_model(input_shape)
dl_models['Simple RNN'] = create_simple_rnn_model(input_shape)

print("Deep learning models created:")
for name in dl_models.keys():
    print(f"• {name}")
    
# Set up early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

Input shape for deep learning models: (12, 1)
Deep learning models created:
• LSTM
• GRU
• CNN-LSTM
• Simple RNN


In [88]:
# Train deep learning models
print("Training deep learning models...")
print("="*50)

dl_predictions = {}
dl_histories = {}
training_times = {}

for model_name, model in dl_models.items():
    print(f"\nTraining {model_name}...")
    
    # Record training time
    import time
    start_time = time.time()
    
    # Train the model
    history = model.fit(
        X_train_dl_reshaped, y_train_dl,
        epochs=100,
        batch_size=8,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=0  # Suppress detailed output
    )
    
    end_time = time.time()
    training_time = end_time - start_time
    training_times[model_name] = training_time
    
    # Make predictions
    predictions_scaled = model.predict(X_test_dl_reshaped, verbose=0)
    
    # Inverse transform predictions back to original scale (now correctly shaped)
    predictions = scaler_dl.inverse_transform(predictions_scaled)
    dl_predictions[model_name] = predictions.flatten()
    dl_histories[model_name] = history
    
    print(f"✓ {model_name} trained in {training_time:.1f} seconds")
    print(f"  Final training loss: {history.history['loss'][-1]:.6f}")
    print(f"  Final validation loss: {history.history['val_loss'][-1]:.6f}")
    print(f"  Epochs trained: {len(history.history['loss'])}")

print(f"\n🎉 All deep learning models trained successfully!")

Training deep learning models...

Training LSTM...
✓ LSTM trained in 7.5 seconds
  Final training loss: 0.020494
  Final validation loss: 0.040955
  Epochs trained: 18

Training GRU...
✓ GRU trained in 28.5 seconds
  Final training loss: 0.017968
  Final validation loss: 0.033412
  Epochs trained: 100

Training CNN-LSTM...
✓ CNN-LSTM trained in 4.4 seconds
  Final training loss: 0.018802
  Final validation loss: 0.045779
  Epochs trained: 10

Training Simple RNN...
✓ Simple RNN trained in 5.3 seconds
  Final training loss: 0.020091
  Final validation loss: 0.034748
  Epochs trained: 18

🎉 All deep learning models trained successfully!


In [93]:
# Evaluate deep learning models
print("Evaluating Deep Learning Models")
print("="*50)

# Get actual test values (inverse transform)
y_test_actual = scaler_dl.inverse_transform(y_test_dl.reshape(-1, 1)).flatten()

# Evaluate deep learning models
dl_results = []
for model_name, predictions in dl_predictions.items():
    mae = mean_absolute_error(y_test_actual, predictions)
    mse = mean_squared_error(y_test_actual, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_actual, predictions)
    mape = np.mean(np.abs((y_test_actual - predictions) / y_test_actual)) * 100
    
    dl_results.append({
        'Model': f'{model_name} (DL)',
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R²': r2,
        'MAPE': mape,
        'Training Time (s)': training_times[model_name]
    })

# Create results DataFrame
dl_results_df = pd.DataFrame(dl_results)
print("Deep Learning Model Performance:")
print("="*60)
print(dl_results_df.round(2))

# Find best deep learning model
best_dl_idx = dl_results_df['RMSE'].idxmin()
best_dl_model = dl_results_df.loc[best_dl_idx, 'Model']
best_dl_rmse = dl_results_df.loc[best_dl_idx, 'RMSE']
best_dl_mape = dl_results_df.loc[best_dl_idx, 'MAPE']

print(f"\n🏆 Best Deep Learning Model: {best_dl_model}")
print(f"   RMSE: ${best_dl_rmse:,.2f}")
print(f"   MAPE: {best_dl_mape:.2f}%")
print(f"   Training Time: {dl_results_df.loc[best_dl_idx, 'Training Time (s)']:.1f} seconds")

print(f"\nDeep Learning vs Best Traditional (Random Forest):")
improvement = ((83403 - best_dl_rmse) / 83403) * 100
print(f"• RMSE improvement: {improvement:+.1f}%")
mape_improvement = ((12.2 - best_dl_mape) / 12.2) * 100
print(f"• MAPE improvement: {mape_improvement:+.1f}%")

Evaluating Deep Learning Models
Deep Learning Model Performance:
             Model      MAE          MSE     RMSE    R²   MAPE  \
0        LSTM (DL)  3376.15  29479611.19  5429.51 -0.05  60.69   
1         GRU (DL)  2953.46  21925049.40  4682.42  0.22  53.64   
2    CNN-LSTM (DL)  3548.74  34255462.63  5852.82 -0.22  49.41   
3  Simple RNN (DL)  3167.61  24216423.35  4921.02  0.13  56.18   

   Training Time (s)  
0               7.46  
1              28.48  
2               4.38  
3               5.27  

🏆 Best Deep Learning Model: GRU (DL)
   RMSE: $4,682.42
   MAPE: 53.64%
   Training Time: 28.5 seconds

Deep Learning vs Best Traditional (Random Forest):
• RMSE improvement: +94.4%
• MAPE improvement: -339.7%
